# 01 — Data Acquisition

| Field | Detail |
|---|---|
| **Notebook** | `01_data_acquisition.ipynb` |
| **Pipeline stage** | H1 — Voice Emotion Recognition - 1 of 7 (Data Acquisition) |
| **Owner(s)** | *Thrithwaka* |
| **Created** | *27/08/2026* |
| **Last updated** | *27/08/2026* |
| **Upstream dependency** | None — this is the entry point of the H1 pipeline |
| **Downstream dependency** | `02_preprocessing.ipynb` reads the outputs of this notebook |
| **Research proposal reference** | Section 5.1 (Voice Emotion Recognition) & Section 6 (Resource Requirements — RAVDESS, TESS datasets); SAVEE added per team decision (see `docs/research_proposal_mapping.md`) |

## Purpose

This notebook downloads, verifies, and catalogs the three raw datasets used for H1 model training and comparison:

- **RAVDESS** — Ryerson Audio-Visual Database of Emotional Speech and Song (speech-only subset)
- **TESS** — Toronto Emotional Speech Set
- **SAVEE** — Surrey Audio-Visual Expressed Emotion

It does **not** perform any label mapping, resampling, or feature extraction — that is the responsibility of `02_preprocessing.ipynb`. Keeping acquisition and preprocessing separate means the raw data can be re-verified independently of any preprocessing logic changes, and every team member can confirm they are working from byte-identical source data via the checksum manifest produced at the end of this notebook.

## Objectives

1. Set up a reproducible, version-controlled directory structure for raw data.
2. Download (or verify manual placement of) RAVDESS, TESS, and SAVEE.
3. Validate file counts against known expected totals for each dataset.
4. Run basic audio-integrity checks (readability, sample rate, duration) on a sample of files.
5. Generate a SHA-256 checksum manifest so every team member can verify they have identical raw data.
6. Produce a machine-readable acquisition report consumed by the next notebook and by `docs/`.
7. Leave a clear, structured handoff note for whoever runs `02_preprocessing.ipynb` next.

## 0. Environment Setup

In [1]:
import os
import sys
import json
import hashlib
import logging
import subprocess
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

# Make the project root importable so we can reuse config/settings.py,
# matching every other notebook and script in this repository.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: C:\Users\thrit\Desktop\emotion-ai-companion-research


In [2]:
# --- Logging setup -----------------------------------------------------
# Every notebook in this pipeline logs to both the console and a shared
# per-notebook log file under reports/logs/, so issues can be diffed
# across team members' runs without re-executing anything.

LOG_DIR = PROJECT_ROOT / "reports" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "01_data_acquisition.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, mode="w"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("data_acquisition")
logger.info("Logging initialized. Log file: %s", LOG_PATH)

2026-08-27 12:19:21,305 | INFO | Logging initialized. Log file: C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\logs\01_data_acquisition.log


## 1. Configuration

All paths and expected values are defined once, here, and reused by every cell below. If a teammate needs to change a raw-data location, this is the only cell that should require editing.

In [3]:
# --- Directory layout ---------------------------------------------------
RAW_DATA_DIR = PROJECT_ROOT / "training" / "data"
MANUAL_DOWNLOADS_DIR = RAW_DATA_DIR / "manual_downloads"   # fallback if Kaggle API is unavailable

DATASET_DIRS = {
    "ravdess": RAW_DATA_DIR / "ravdess_raw",
    "tess": RAW_DATA_DIR / "tess_raw",
    "savee": RAW_DATA_DIR / "savee_raw",
}

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Kaggle dataset identifiers ------------------------------------------
# These map to the 'owner/dataset-slug' identifiers used by the Kaggle API.
# Source links are also documented in docs/research_proposal_mapping.md.
KAGGLE_DATASETS = {
    "ravdess": "uwrfkaggler/ravdess-emotional-speech-audio",
    "tess": "ejlok1/toronto-emotional-speech-set-tess",
    "savee": "ejlok1/surrey-audiovisual-expressed-emotion-savee",
}

# --- Expected file counts (used for validation in Section 3) ------------
# RAVDESS: 1440 SPEECH files (excludes the ~1012 song files, which are out
#          of scope per the proposal's speech-only focus).
# TESS:    2800 utterances (7 emotions x 400 files, 2 speakers).
# SAVEE:   480 utterances (7 emotions, 4 male speakers).
EXPECTED_FILE_COUNTS = {
    "ravdess": 1440,
    "tess": 2800,
    "savee": 480,
}

AUDIO_EXTENSIONS = {".wav"}

# --- Reproducibility ------------------------------------------------------
ACQUIRED_BY = "CHANGE_ME"          # set to your name/GitHub handle before running
ACQUISITION_DATE = datetime.now(timezone.utc).isoformat()

for path in list(DATASET_DIRS.values()) + [MANUAL_DOWNLOADS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

logger.info("Configuration loaded. Datasets: %s", list(DATASET_DIRS.keys()))

2026-08-27 12:19:24,002 | INFO | Configuration loaded. Datasets: ['ravdess', 'tess', 'savee']


## 2. Dataset Acquisition

Two supported paths are provided, since Kaggle requires authenticated API access and not every team member's machine (or the shared Colab environment) will have credentials configured identically:

**Path A — Kaggle API (preferred, fully automated).**
Requires a `kaggle.json` API token. To obtain one: Kaggle account settings → *API* → *Create New Token*. Place the downloaded file at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\Users\<you>\.kaggle\kaggle.json` (Windows), then run `chmod 600 ~/.kaggle/kaggle.json` on Linux/Mac.

**Path B — Manual download fallback.**
If the Kaggle API is unavailable (e.g. institutional network restrictions), download the three dataset zip files manually from the Kaggle URLs in `docs/research_proposal_mapping.md`, and place them, unmodified, in `training/data/manual_downloads/` using the exact filenames printed by the cell below. The extraction cell will detect and use them automatically.

**Both paths converge**: after this section, all three datasets end up extracted under `training/data/{ravdess,tess,savee}_raw/`, regardless of which path was used.

In [4]:
def kaggle_api_available() -> bool:
    """Checks whether the kaggle CLI is installed and credentials are configured."""
    try:
        import kaggle  # noqa: F401
    except (ImportError, OSError) as exc:
        logger.warning("Kaggle package not usable: %s", exc)
        return False
    cred_path = Path.home() / ".kaggle" / "kaggle.json"
    if not cred_path.exists():
        logger.warning("No credentials found at %s", cred_path)
        return False
    return True


USE_KAGGLE_API = kaggle_api_available()
print(f"Kaggle API available: {USE_KAGGLE_API}")
if not USE_KAGGLE_API:
    print("\nFalling back to manual download path. Expected files in", MANUAL_DOWNLOADS_DIR, ":")
    for name, slug in KAGGLE_DATASETS.items():
        print(f"  - {name}.zip   (source: https://www.kaggle.com/datasets/{slug})")

2026-08-27 12:19:29,192 | WARNING | Kaggle package not usable: No module named 'kaggle'
Kaggle API available: False

Falling back to manual download path. Expected files in C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\manual_downloads :
  - ravdess.zip   (source: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio)
  - tess.zip   (source: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess)
  - savee.zip   (source: https://www.kaggle.com/datasets/ejlok1/surrey-audiovisual-expressed-emotion-savee)


In [5]:
def download_via_kaggle(name: str, slug: str, dest_dir: Path) -> Path:
    """Downloads and unzips a Kaggle dataset via the official kaggle CLI."""
    zip_path = dest_dir.parent / f"{name}.zip"
    logger.info("Downloading %s from Kaggle (%s)...", name, slug)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest_dir.parent), "--force"],
        check=True,
    )
    downloaded_zip = dest_dir.parent / f"{slug.split('/')[-1]}.zip"
    if downloaded_zip.exists():
        zip_path = downloaded_zip
    return zip_path


def extract_zip(zip_path: Path, dest_dir: Path) -> None:
    logger.info("Extracting %s -> %s", zip_path.name, dest_dir)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir)


for name, dest_dir in DATASET_DIRS.items():
    existing_count = sum(1 for _ in dest_dir.rglob("*.wav"))
    if existing_count >= EXPECTED_FILE_COUNTS[name]:
        logger.info("%s already extracted (%d files found) — skipping download.", name, existing_count)
        continue

    if USE_KAGGLE_API:
        zip_path = download_via_kaggle(name, KAGGLE_DATASETS[name], dest_dir)
    else:
        zip_path = MANUAL_DOWNLOADS_DIR / f"{name}.zip"
        if not zip_path.exists():
            logger.error(
                "Manual zip not found for '%s' at %s. Download it and re-run this cell.",
                name, zip_path,
            )
            continue

    extract_zip(zip_path, dest_dir)

print("Acquisition step complete. See log for details:", LOG_PATH)

2026-08-27 12:19:30,203 | INFO | ravdess already extracted (1440 files found) — skipping download.
2026-08-27 12:19:30,234 | INFO | tess already extracted (2800 files found) — skipping download.
2026-08-27 12:19:30,240 | INFO | savee already extracted (480 files found) — skipping download.
Acquisition step complete. See log for details: C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\logs\01_data_acquisition.log


## 3. File Count Verification

Compares the number of `.wav` files actually present against the known expected totals for each dataset. A mismatch here almost always means either a partial download/extraction, or that a dataset's internal folder structure changed and the glob pattern needs adjusting — investigate before proceeding to preprocessing.

In [6]:
def count_audio_files(directory: Path) -> int:
    return sum(1 for f in directory.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS)


verification_rows = []
for name, dest_dir in DATASET_DIRS.items():
    actual = count_audio_files(dest_dir)
    expected = EXPECTED_FILE_COUNTS[name]
    status = "OK" if actual == expected else "MISMATCH"
    verification_rows.append({
        "dataset": name,
        "expected_files": expected,
        "actual_files": actual,
        "status": status,
    })
    log_fn = logger.info if status == "OK" else logger.warning
    log_fn("%s: expected=%d actual=%d status=%s", name, expected, actual, status)

verification_df = pd.DataFrame(verification_rows)
verification_df

2026-08-27 12:19:32,321 | INFO | ravdess: expected=1440 actual=1440 status=OK
2026-08-27 12:19:32,353 | INFO | tess: expected=2800 actual=2800 status=OK
2026-08-27 12:19:32,358 | INFO | savee: expected=480 actual=480 status=OK


,dataset,expected_files,actual_files,status
0,ravdess,1440,1440,OK
1,tess,2800,2800,OK
2,savee,480,480,OK


> **If any dataset shows `MISMATCH`:** do not proceed to `02_preprocessing.ipynb` until resolved. Common causes: (1) an incomplete download — re-run the acquisition cell above; (2) the dataset's folder structure differs from what the count function expects (some Kaggle re-uploads nest files an extra directory level deep) — inspect `training/data/<name>_raw/` manually and adjust the path if needed; (3) for RAVDESS specifically, remember the *song* files are intentionally excluded from the 1440 expected count.

## 4. Audio Integrity Sampling

Rather than opening all ~4,700 files (slow, and unnecessary at this stage), this section randomly samples a subset per dataset and verifies each file is readable and has plausible properties. This catches corrupted downloads or truncated files early, before they cause a cryptic failure deep inside a training loop in a later notebook.

In [7]:
import sys

print(sys.executable)

C:\Users\thrit\Desktop\emotion-ai-companion-research\venv\Scripts\python.exe


In [8]:
import soundfile as sf

print("SoundFile version:", sf.__version__)
print("SoundFile imported successfully!")

SoundFile version: 0.12.1
SoundFile imported successfully!


In [9]:
import random
import soundfile as sf

random.seed(42)  # fixed seed — matches the project-wide seed convention (see config/settings.py)
SAMPLE_SIZE_PER_DATASET = 30

integrity_rows = []
for name, dest_dir in DATASET_DIRS.items():
    all_files = [f for f in dest_dir.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS]
    sample = random.sample(all_files, min(SAMPLE_SIZE_PER_DATASET, len(all_files)))

    for filepath in sample:
        try:
            data, sr = sf.read(str(filepath))
            duration_s = len(data) / sr
            integrity_rows.append({
                "dataset": name,
                "file": filepath.name,
                "readable": True,
                "sample_rate": sr,
                "duration_s": round(duration_s, 3),
            })
        except Exception as exc:
            logger.error("Corrupt or unreadable file: %s (%s)", filepath, exc)
            integrity_rows.append({
                "dataset": name,
                "file": filepath.name,
                "readable": False,
                "sample_rate": None,
                "duration_s": None,
            })

integrity_df = pd.DataFrame(integrity_rows)
integrity_df

,dataset,file,readable,sample_rate,duration_s
0,ravdess,03-01-07-02-01-02-22.wav,True,48000,4.171
1,ravdess,03-01-07-02-01-01-04.wav,True,48000,4.004
2,ravdess,03-01-07-02-02-02-01.wav,True,48000,4.371
3,ravdess,03-01-04-01-02-02-10.wav,True,48000,3.804
4,ravdess,03-01-04-01-01-02-09.wav,True,48000,3.437
...,...,...,...,...,...
85,savee,JK_sa02.wav,True,44100,4.291
86,savee,DC_f07.wav,True,44100,3.616
87,savee,JK_n12.wav,True,44100,3.276
88,savee,DC_n28.wav,True,44100,5.499


In [10]:
# Summary statistics per dataset — flags anything unexpected (e.g. wildly
# inconsistent sample rates within one dataset, which would need explicit
# handling in the resampling step of 02_preprocessing.ipynb).
summary = integrity_df.groupby("dataset").agg(
    files_checked=("file", "count"),
    all_readable=("readable", "all"),
    sample_rates_seen=("sample_rate", lambda s: sorted(s.dropna().unique().tolist())),
    min_duration_s=("duration_s", "min"),
    max_duration_s=("duration_s", "max"),
)
summary

,files_checked,all_readable,sample_rates_seen,min_duration_s,max_duration_s
dataset,,,,,
ravdess,30,True,[48000],3.270,4.371
savee,30,True,[44100],1.855,6.289
tess,30,True,[24414],1.457,2.646


## 5. Checksum Manifest

Generates a SHA-256 hash for every raw audio file. This manifest serves two purposes for a team of four working across different machines and a shared Colab environment:

1. **Verification** — any teammate can hash their local copy and diff it against `data/raw/checksums.json` to confirm they have byte-identical data before reporting a training discrepancy as a 'bug'.
2. **Change detection** — if a dataset is ever re-downloaded (e.g. after a Kaggle re-upload), this manifest immediately shows exactly which files changed.

In [11]:
def sha256_of_file(path: Path, chunk_size: int = 8192) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


checksum_manifest = {}
for name, dest_dir in DATASET_DIRS.items():
    files = sorted(f for f in dest_dir.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS)
    logger.info("Hashing %d files for %s...", len(files), name)
    checksum_manifest[name] = {
        str(f.relative_to(PROJECT_ROOT)): sha256_of_file(f) for f in files
    }

checksum_path = RAW_DATA_DIR / "checksums.json"
with open(checksum_path, "w") as f:
    json.dump(checksum_manifest, f, indent=2)

total_hashed = sum(len(v) for v in checksum_manifest.values())
logger.info("Wrote checksum manifest for %d files to %s", total_hashed, checksum_path)
print(f"Checksum manifest written: {checksum_path} ({total_hashed} files)")

2026-08-27 12:21:09,122 | INFO | Hashing 1440 files for ravdess...
2026-08-27 12:21:24,248 | INFO | Hashing 2800 files for tess...
2026-08-27 12:21:50,038 | INFO | Hashing 480 files for savee...
2026-08-27 12:21:54,964 | INFO | Wrote checksum manifest for 4720 files to C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\checksums.json
Checksum manifest written: C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\checksums.json (4720 files)


## 6. Acquisition Report

A single machine-readable summary of everything this notebook did, consumed by `02_preprocessing.ipynb` (to confirm it's reading verified data) and referenced in the thesis methodology section for an auditable record of dataset provenance.

In [12]:
acquisition_report = {
    "acquired_by": ACQUIRED_BY,
    "acquisition_timestamp_utc": ACQUISITION_DATE,
    "kaggle_api_used": USE_KAGGLE_API,
    "datasets": {
        name: {
            "source": f"https://www.kaggle.com/datasets/{KAGGLE_DATASETS[name]}",
            "local_path": str(DATASET_DIRS[name].relative_to(PROJECT_ROOT)),
            "expected_files": EXPECTED_FILE_COUNTS[name],
            "actual_files": int(
                verification_df.loc[verification_df["dataset"] == name, "actual_files"].iloc[0]
            ),
            "verification_status": verification_df.loc[
                verification_df["dataset"] == name, "status"
            ].iloc[0],
        }
        for name in DATASET_DIRS
    },
    "checksum_manifest_path": str(checksum_path.relative_to(PROJECT_ROOT)),
    "integrity_sample_size_per_dataset": SAMPLE_SIZE_PER_DATASET,
    "all_sampled_files_readable": bool(integrity_df["readable"].all()),
}

report_path = REPORTS_DIR / "01_data_acquisition_report.json"
with open(report_path, "w") as f:
    json.dump(acquisition_report, f, indent=2)

logger.info("Acquisition report written to %s", report_path)
print(f"Acquisition report written: {report_path}")
print(json.dumps(acquisition_report, indent=2))

2026-08-27 12:22:13,175 | INFO | Acquisition report written to C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\01_data_acquisition_report.json
Acquisition report written: C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\01_data_acquisition_report.json
{
  "acquired_by": "CHANGE_ME",
  "acquisition_timestamp_utc": "2026-08-27T06:49:24.001429+00:00",
  "kaggle_api_used": false,
  "datasets": {
    "ravdess": {
      "source": "https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio",
      "local_path": "training\\data\\ravdess_raw",
      "expected_files": 1440,
      "actual_files": 1440,
      "verification_status": "OK"
    },
    "tess": {
      "source": "https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess",
      "local_path": "training\\data\\tess_raw",
      "expected_files": 2800,
      "actual_files": 2800,
      "verification_status": "OK"
    },
    "savee": {
      "source": "https://www.kaggle.com/datasets/ej

## 7. Sanity Check — Final Directory Snapshot

A quick human-readable view of the final state, useful to paste into a PR description or a team chat when handing this off.

In [13]:
print("Final raw data directory structure:\n")
for name, dest_dir in DATASET_DIRS.items():
    n_files = count_audio_files(dest_dir)
    size_mb = sum(f.stat().st_size for f in dest_dir.rglob("*") if f.is_file()) / (1024 ** 2)
    print(f"  {dest_dir.relative_to(PROJECT_ROOT)}/")
    print(f"      files: {n_files}")
    print(f"      size:  {size_mb:.1f} MB\n")

print(f"Checksum manifest: {checksum_path.relative_to(PROJECT_ROOT)}")
print(f"Acquisition report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Log file: {LOG_PATH.relative_to(PROJECT_ROOT)}")

Final raw data directory structure:

  training\data\ravdess_raw/
      files: 1440
      size:  563.0 MB

  training\data\tess_raw/
      files: 2800
      size:  268.3 MB

  training\data\savee_raw/
      files: 480
      size:  155.0 MB

Checksum manifest: training\data\checksums.json
Acquisition report: reports\01_data_acquisition_report.json
Log file: reports\logs\01_data_acquisition.log


---

## Summary Note — Handoff to Next Notebook

*(Auto-drafted structure below — fill in the italicized placeholders with your actual run's values before committing, and update the `ACQUIRED_BY` / dates. This section is intended to be read by a teammate who has never opened this notebook before.)*

### What was done in this notebook

1. Established the raw-data directory layout under `training/data/{ravdess,tess,savee}_raw/`.
2. Downloaded RAVDESS, TESS, and SAVEE (via Kaggle API, or manual fallback — see Section 2 for which path was used this run).
3. Verified file counts against known expected totals (1440 / 2800 / 480 respectively) — see Section 3 output for this run's actual counts.
4. Sampled 30 files per dataset and confirmed they are readable, logging sample rate and duration statistics — see Section 4.
5. Generated a SHA-256 checksum for every raw audio file, so all teammates can verify they're working from identical data.
6. Wrote a machine-readable acquisition report summarizing all of the above.

### Outputs produced by this notebook (and where to find them)

| Output | Location | Used by |
|---|---|---|
| Raw RAVDESS audio | `training/data/ravdess_raw/` | `02_preprocessing.ipynb` |
| Raw TESS audio | `training/data/tess_raw/` | `02_preprocessing.ipynb` |
| Raw SAVEE audio | `training/data/savee_raw/` | `02_preprocessing.ipynb` |
| Checksum manifest | `training/data/checksums.json` | Any teammate verifying their local data matches the team's canonical copy |
| Acquisition report | `reports/01_data_acquisition_report.json` | `02_preprocessing.ipynb` (sanity check at its own startup); thesis methodology section (dataset provenance) |
| Run log | `reports/logs/01_data_acquisition.log` | Debugging if `02_preprocessing.ipynb` reports missing/unexpected files |

### What needs to be done next

Proceed to **`02_preprocessing.ipynb`**, which will:

- Read the three raw directories produced here.
- Map each dataset's native emotion labels onto the project's unified 6-class scheme (happy, sad, angry, fear, neutral, surprise) — see `docs/research_proposal_mapping.md` for the exact mapping table, since RAVDESS/TESS/SAVEE do not use identical label vocabularies.
- Resample all audio to a single common sample rate and standardize clip duration.
- Produce a single unified, stratified train/val/test manifest (`data/processed/h1_manifest.csv`) that every subsequent H1 notebook (feature extraction, all four model-training notebooks, and model selection) will read from.

**Before running `02_preprocessing.ipynb`, confirm:**
- Section 3's verification table shows `OK` for all three datasets (re-run Section 2 if not).
- Section 4's `all_readable` check is `True` for all sampled files.
- `reports/01_data_acquisition_report.json` exists and its `verification_status` fields all read `"OK"`.

### Resources needed for the next step

| Resource | Needed for | Notes |
|---|---|---|
| `librosa`, `soundfile` (already in `requirements.txt`) | Resampling and feature extraction in `02_preprocessing.ipynb` | No new installs required |
| `docs/research_proposal_mapping.md` | Label-mapping table between each dataset's native emotions and the unified 6-class scheme | Read this before writing the mapping dictionary in `02_preprocessing.ipynb` |
| `config/settings.py` (`emotion_labels` list) | Ensures the manifest's label set matches what every model module (`src/emotion/*.py`) expects | Do not hardcode a separate label list in the notebook — import from here |
| This notebook's checksum manifest | Optional integrity re-check if another teammate re-clones the raw data independently | `training/data/checksums.json` |

### Known issues / things to watch for

- *(Fill in anything discovered during this run — e.g. "SAVEE's Kaggle mirror nests files one directory deeper than expected" or "had to use manual download path because Kaggle API rate-limited on campus wifi".)*

### Run metadata

- **Acquired by:** *Thrithwaka*
- **Date:** *27/08/2026*
- **Kaggle API used:** *No*
- **All verification checks passed:** *Yes*